# Исходный пример BlocksNet
Источник: https://github.com/aimclub/blocksnet/blob/main/examples/pipeline.ipynb
Справочный материал. Это не результаты исследования Казани. Выводы ячеек удалены.


# Pipeline

## 1. Blocks cutting

### 1.1. Fetching data from OpenStreetMap. 
The step can be skipped, if the data already exists.

In [ ]:
import osmnx as ox

boundaries = ox.geocode_to_gdf('Санкт-Петербург, Василеостровский район')
boundaries.plot().set_axis_off()

In [ ]:
bc_tags = {
    'roads': {
      "highway": ["construction","crossing","living_street","motorway","motorway_link","motorway_junction","pedestrian","primary","primary_link","raceway","residential","road","secondary","secondary_link","services","tertiary","tertiary_link","track","trunk","trunk_link","turning_circle","turning_loop","unclassified",],
      "service": ["living_street", "emergency_access"]
    },
    'railways': {
      "railway": "rail"
    },
    'water': {
      'riverbank':True,
      'reservoir':True,
      'basin':True,
      'dock':True,
      'canal':True,
      'pond':True,
      'natural':['water','bay'],
      'waterway':['river','canal','ditch'],
      'landuse':'basin',
      'water': 'lake'
    }
}

In [ ]:
water = ox.features_from_polygon(boundaries.union_all(), bc_tags['water'])
roads = ox.features_from_polygon(boundaries.union_all(), bc_tags['roads'])
railways = ox.features_from_polygon(boundaries.union_all(), bc_tags['railways'])

In [ ]:
water = water[water.geom_type.isin(['Polygon', 'MultiPolygon', 'LineString', 'MultiLineString'])].copy()
roads = roads[roads.geom_type.isin(['LineString', 'MultiLineString'])].copy()
railways = railways[railways.geom_type.isin(['LineString', 'MultiLineString'])].copy()

In [ ]:
crs = boundaries.estimate_utm_crs()

for gdf in [water, roads, railways, boundaries]:
  gdf.to_crs(crs, inplace=True)

### 1.2. Preprocessing input geometry

The step can be skipped if input geometries are already sorted as lines, polygons and boundary.

In [ ]:
from blocksnet.blocks.cutting import preprocess_urban_objects, cut_urban_blocks

lines, polygons = preprocess_urban_objects(roads, railways, water)

### 1.3. Cutting urban blocks

In [ ]:
blocks = cut_urban_blocks(boundaries, lines, polygons)

In [ ]:
blocks.plot().set_axis_off()

## 2. Land use assignment

### 2.1. Fetching data from OpenStreetMap

The step can be skipped if functional zones layer is already obtained via goverment sources.

In [ ]:
import osmnx as ox

functional_zones = ox.features_from_polygon(boundaries.to_crs(4326).union_all(), tags={'landuse': True})

In [ ]:
functional_zones = functional_zones.reset_index(drop=True)[['geometry','landuse']].rename(columns={'landuse': 'functional_zone'})
functional_zones = functional_zones.to_crs(crs)
functional_zones.head()

### 2.2. Specifying rules

Rules define how `functional_zone` column will be mapped into `LandUse`.

In [ ]:
from blocksnet.enums import LandUse

rules = {
  'commercial': LandUse.BUSINESS,
  'industrial': LandUse.INDUSTRIAL,
  'cemetery': LandUse.SPECIAL,
  'garages': LandUse.INDUSTRIAL,
  'residential': LandUse.RESIDENTIAL,
  'retail': LandUse.BUSINESS,
  'grass': LandUse.RECREATION,
  'farmland': LandUse.AGRICULTURE,
  'construction': LandUse.SPECIAL,
  'brownfield': LandUse.INDUSTRIAL,
  'forest': LandUse.RECREATION,
  'recreation_ground': LandUse.RECREATION,
  'religious': LandUse.SPECIAL,
  'flowerbed': LandUse.RECREATION,
  'military': LandUse.SPECIAL,
  'landfill': LandUse.TRANSPORT
}

### 2.3. Assignment

In [ ]:
from blocksnet.blocks.assignment import assign_land_use

blocks = assign_land_use(blocks, functional_zones.reset_index(drop=True).rename(columns={'landuse': 'functional_zone'}), rules)

In [ ]:
blocks.head()

In [ ]:
ax = blocks.plot(color='#ddd')
blocks.plot(column='land_use', legend=True, ax=ax).set_axis_off()

## 3. Accessibility matrix calculation

There are few ways to define spatial relations between urban blocks:
- **Accessibility matrix** -- `n^2` matrix that defines travel time between urban blocks' centroids in minutes.
- **Distance matrix** -- `n^2` matrix that defines euclidian distance between urban blocks' centroids in meters.
- **Adjacency graph** -- graph with `n` nodes. Each edge of this graph indicates spatial adjacency of two urban blocks.

In this example the **accessibility matrix** will be calculated using `IduEdu` library.

### 3.1. Intermodal graph construction

There are few types of city graphs to model city network:
- **walk** -- pedestrian network.
- **drive** -- personal transport network.
- **intermodal** -- pedestrian and public transport network.

In this example **intermodal** graph will be constructed, but overall it depends on one's research.

In [ ]:
from blocksnet.relations import get_accessibility_graph

graph = get_accessibility_graph(boundaries, 'intermodal')

### 3.2. Calculating the accessibility matrix

In [ ]:
from blocksnet.relations import calculate_accessibility_matrix

acc_mx = calculate_accessibility_matrix(blocks, graph)

In [ ]:
acc_mx.head()

## 4. Buildings aggregation

Buildings define the urban environment parameters in different blocks. For some blocksnet methods only population can be used, but overall the building is defined via:

- `population : float`
- `footprint_area : float` -- building's base area
- `number_of_floors : float`
- `build_floor_area : float` -- sum area of each building floor
- `is_living : bool`
- `living_area : float` -- area defined for residents
- `non_living_area : float` -- area not defined to live in

Some of these parameters may be optional, depends on what one already has.

### 4.1. Fetching data from OpenStreetMap

The step can be skipped if the data already obtained from any available source.

In [ ]:
buildings = ox.features_from_polygon(boundaries.to_crs(4326).union_all(), tags={'building': True}).reset_index(drop=True).to_crs(crs)

#### 4.1.1. `is_living`

In [ ]:
is_living_tags = ['residential', 'house', 'apartments', 'detached', 'terrace', 'dormitory']
buildings['is_living'] = buildings['building'].apply(lambda b : b in is_living_tags)

#### 4.1.2. `number_of_floors`

In [ ]:
import pandas as pd

buildings['number_of_floors'] = pd.to_numeric(buildings['building:levels'], errors='coerce')

### 4.2. Imputing missing data

In [ ]:
from blocksnet.preprocessing.imputing import impute_buildings

buildings = impute_buildings(buildings, default_living_demand=30)

In [ ]:
buildings.sample(5)

In [ ]:
buildings.population.sum()

### 4.3. Aggregating buildings within blocks

In [ ]:
from blocksnet.blocks.aggregation import aggregate_objects

buildings_blocks = aggregate_objects(blocks, buildings)[0]

In [ ]:
buildings_blocks.head()

In [ ]:
buildings_blocks.plot('population', legend=True).set_axis_off()

## 5. Services aggregation

### 5.1. Fetching data from OpenStreetMap

And once again: if one already has the required data, the step can be skipped.

In [ ]:
schools = ox.features_from_polygon(boundaries.to_crs(4326).union_all(), tags={'amenity': 'school'}).reset_index(drop=True).to_crs(crs)

### 5.2. Imputing missing data

In [ ]:
from blocksnet.preprocessing.imputing import impute_services

schools = impute_services(schools, 'school')

In [ ]:
schools.head()

### 5.3. Aggregating services within blocks

In [ ]:
schools_blocks = aggregate_objects(blocks, schools)[0]

In [ ]:
schools_blocks.head()

In [ ]:
schools_blocks.plot('capacity', legend=True).set_axis_off()

## 6. Results

Using the methods above the following data can be obtained:

- **Urban blocks layer** -- `GeoDataFrame` of urban blocks within the modeled territory (city).
- **Land use assignment layer** -- the assignment of current land use according to any source of functional zones data.
- **Accessibility matrix** -- `DataFrame` `n*n` matrix of accessibility values in minutes between each pair of urban blocks.
- **Buildings aggregation layer** -- the aggregation of buildings parameters within urban blocks.
- **Services aggregation layer** -- the aggregation of services parameters within urban blocks.

It's important to mention that any type of data can be aggregated within city blocks depending on the task.

## 7. Use examples

### 7.1. Competitive provision

In [ ]:
df = buildings_blocks[['population']].join(schools_blocks[['capacity']])
df.head()

In [ ]:
from blocksnet.config import service_types_config

_, demand, accessibility = service_types_config['school'].values()
demand, accessibility

In [ ]:
from blocksnet.analysis.provision import competitive_provision

prov_df, links_df = competitive_provision(df, acc_mx, accessibility, demand)

In [ ]:
prov_df.head()

In [ ]:
links_df.head()

In [ ]:
ax = blocks.plot(color='#ddd')
blocks.join(prov_df).plot('provision_strong', ax=ax, cmap='RdYlGn', vmin=0, vmax=1, legend=True)
ax.set_axis_off()

### 7.2. Morphotypes

In [ ]:
from blocksnet.analysis.indicators import calculate_density_indicators

buildings_blocks['site_area'] = buildings_blocks.area
density_df = calculate_density_indicators(buildings_blocks)
density_df.head()

#### 7.2.1. Spacematrix morphotypes

In [ ]:
from blocksnet.analysis.morphotypes import get_spacematrix_morphotypes

spacematrix_df, spacematrix_clusters = get_spacematrix_morphotypes(density_df)

In [ ]:
spacematrix_df.head()

In [ ]:
spacematrix_clusters.head()

In [ ]:
ax = blocks.plot(color='#ddd')
blocks.join(spacematrix_df).plot('morphotype', legend=True, ax=ax)
ax.set_axis_off()

#### 7.2.2. Strelka morphotypes

In [ ]:
from blocksnet.analysis.morphotypes import get_strelka_morphotypes

strelka_df = get_strelka_morphotypes(density_df)

In [ ]:
strelka_df.head()

In [ ]:
ax = blocks.plot(color='#ddd')
blocks.join(strelka_df).plot('morphotype', legend=True, ax=ax)
ax.set_axis_off()

### 7.3. Accessibility analysis

In [ ]:
from blocksnet.analysis.accessibility import area_accessibility

area_acc = area_accessibility(acc_mx, buildings_blocks)

In [ ]:
blocks.join(area_acc).plot('area_accessibility', legend=True).set_axis_off()